## Workflow Overview  
**Anomalous Sightings Archive Project**

### Tools & Environment
- **Version Control**: GitHub
- **IDE**: VS Code
- **Primary Development**: Jupyter Notebooks (data wrangling, database creation, and visualizations)
- **Modular Python Scripts** (in `python/`):
    - Weather API integration
    - KP Index API fetching and CSV generation
    - Geohash generation from latitude/longitude
    - Proximity table computation and distance enrichment

### Project Workflow

1. **Data Ingestion**  
   Load original datasets into the `data/` directory:  
   - Bigfoot reports (2 BFRO datasets from Kaggle)  
   - UAP reports (NUFORC dataset from Kaggle)  
   - US Census 2010 state population data  
   - Generate `kp_index.csv` via dedicated API script

2. **Data Cleaning** (Pandas)  
   - Standard cleaning and normalization  
   - Merge the two Bigfoot datasets into `combined_bigfoot.csv`  
   - Save cleaned DataFrames as CSVs for backup and reproducibility

3. **Data Enrichment** (Pandas)  
   - Add solar KP Index and AP Index to relevant DataFrames  
   - Apply historical weather data (2010–2014 UAP reports) using `weather_api.py`  
     *(Full UAP dataset spans 1940–2014; Bigfoot data already contains weather)*  
   - Create proximity table by merging UAP and Bigfoot records on `geohash_7`  
   - Reorder columns to align with the Entity Relationship Diagram (ERD)

4. **Relational Database Creation** (SQLite)  
   Build the database with the following tables:  
   - `bigfoot_reports` (PK: `bf_id`)  
   - `uap_reports` (PK: `uap_id`)  
   - `states` (PK: `state_code`)  
   - `proximity` (composite PK: `bf_id` + `uap_id`)  
   - `kp_index` (PK: `datetime`)

5. **Analysis**  
   Execute SQL queries against the SQLite database to generate insights for visualization.

6. **Visualizations**  
   - Charts: Matplotlib + Seaborn  
   - Maps: GeoPandas + Folium

7. **Front-End & Stretch Goals**  
   - Interactive dashboard using **Streamlit** (primary option) or a static site  
   - User experience submission form (Python script / Streamlit page)

In [14]:
# imports 

import pandas as pd
import datetime
import pygeohash as pgh
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import sys
import time 
import requests
import dotenv
from pathlib import Path
import sqlite3

from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv


root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from python.geo_location import create_geohashes

In [15]:
# uap with weather - remove index

uap_2010_wx_df = pd.read_csv("../data/processed/sighting_with_weather_v2 copy.csv")
uap_2010_wx_df = uap_2010_wx_df.drop(columns='Unnamed: 0')
uap_2010_wx_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str,weather_cloud,weather_temp_f,weather_condition,condition_groups
0,2010-10-10 01:00:00,orchard park,ny,us,light,7200.0,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,1,8.0,43.3,Clear,Clear
1,2010-10-10 02:30:00,harrisburg,pa,us,circle,240.0,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,2,0.0,48.5,Clear,Clear
2,2010-10-10 03:00:00,euclid,oh,us,circle,180.0,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,3,8.0,52.2,Clear,Clear
3,2010-10-10 08:30:00,starr,sc,us,formation,600.0,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,8,0.0,64.3,Sunny,Clear
4,2010-10-10 10:45:00,leominster,ma,us,flash,0.0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11,3.0,59.4,Sunny,Clear


In [16]:
us_uap_1940_df = pd.read_csv("../data/processed/us_uap_1940_v1.csv")

uap_2010_wx_df = pd.merge(
    uap_2010_wx_df,
    us_uap_1940_df[["uap_id", "state_code", "datetime", "city"]],
    on=["datetime", "city"],
    how="left"  
)

uap_2010_wx_df["uap_id"] = uap_2010_wx_df["uap_id"].astype('Int64')

uap_2010_wx_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str,weather_cloud,weather_temp_f,weather_condition,condition_groups,uap_id,state_code
0,2010-10-10 01:00:00,orchard park,ny,us,light,7200.0,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,1,8.0,43.3,Clear,Clear,49443,NY
1,2010-10-10 02:30:00,harrisburg,pa,us,circle,240.0,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,2,0.0,48.5,Clear,Clear,49444,PA
2,2010-10-10 03:00:00,euclid,oh,us,circle,180.0,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,3,8.0,52.2,Clear,Clear,49445,OH
3,2010-10-10 08:30:00,starr,sc,us,formation,600.0,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,8,0.0,64.3,Sunny,Clear,49446,SC
4,2010-10-10 10:45:00,leominster,ma,us,flash,0.0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11,3.0,59.4,Sunny,Clear,49447,MA


In [17]:
uap_2010_wx_df = uap_2010_wx_df[[
    'uap_id','datetime', 'city', 'state_code', 'state', 'country', 
    'shape', 'duration (seconds)', 'duration (hours/min)', 'comments', 'date posted', 
    'latitude','longitude', 'rounded_dt', 'date_str', 'hour_str', 
    'weather_cloud','weather_temp_f', 'weather_condition', 'condition_groups'
    ]]

uap_2010_wx_df.head()

,uap_id,datetime,city,state_code,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,rounded_dt,date_str,hour_str,weather_cloud,weather_temp_f,weather_condition,condition_groups
0,49443,2010-10-10 01:00:00,orchard park,NY,ny,us,light,7200.0,a few hours,Xmas colored rotating lights. ((NUFORC Note: ...,1/5/2011,42.7675000,-78.744167,2010-10-10 01:00:00,2010-10-10,1,8.0,43.3,Clear,Clear
1,49444,2010-10-10 02:30:00,harrisburg,PA,pa,us,circle,240.0,4 minutes,possible UFO sighting,11/21/2010,40.2736111,-76.884722,2010-10-10 02:00:00,2010-10-10,2,0.0,48.5,Clear,Clear
2,49445,2010-10-10 03:00:00,euclid,OH,oh,us,circle,180.0,3 minutes,2 objects blinking red and white&#44 disappear...,11/21/2010,41.5930556,-81.526944,2010-10-10 03:00:00,2010-10-10,3,8.0,52.2,Clear,Clear
3,49446,2010-10-10 08:30:00,starr,SC,sc,us,formation,600.0,5-10 mins,Strange orange lights in the night sky,11/21/2010,34.3769444,-82.695833,2010-10-10 08:00:00,2010-10-10,8,0.0,64.3,Sunny,Clear
4,49447,2010-10-10 10:45:00,leominster,MA,ma,us,flash,0.0,NaN,we were in the car and i looked out the window...,11/21/2010,42.5250000,-71.760278,2010-10-10 11:00:00,2010-10-10,11,3.0,59.4,Sunny,Clear


In [18]:
uap_2010_wx_df.to_csv("../data/final/us_uap_2010_2014_weather.csv", index=False)

In [19]:
"""
add renamed csvs to final/ 

us_bigfoot_reports
us_uap_reports
proximity
states
"""

df = pd.read_csv("../data/processed/combined_bigfoot_v1.csv")
df.to_csv("../data/final/bigfoot_reports.csv", index=False)

df = pd.read_csv("../data/processed/us_uap_1940_v1.csv")
df.to_csv("../data/final/uap_reports.csv", index=False)

df = pd.read_csv("../data/processed/proximity_v1.csv")
df.to_csv("../data/final/proximity.csv", index=False)

df = pd.read_csv("../data/processed/states_2010_population_data.csv")
df.to_csv("../data/final/states.csv", index=False)

## Database Setup

In [20]:
# Database Path and Connection
db_path = Path("../data/sql/")
connection = sqlite3.connect(db_path / "anomalous_sightings.db")

# US UAP Reports Table
uap_df = pd.read_csv("../data/final/uap_reports.csv")

uap_df.to_sql(
    name='uap_reports', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'uap_id': 'INTEGER',
        'datetime': 'TEXT',
        'city': 'TEXT',
        'state_code': 'TEXT',
        'state': 'TEXT',
        'duration_secs': 'INTEGER',
        'latitude': 'REAL',
        'longitude': 'REAL',
        'geohash_5': 'TEXT',
        'geohash_6': 'TEXT',
        'geohash_7': 'TEXT',
        'full_date': 'TEXT',
        'year': 'INTEGER',
        'month': 'INTEGER',
        'season': 'TEXT',
        'datetime_formatted': 'TEXT',
        'shape': 'TEXT',
        'shape_group': 'TEXT',
        'comments': 'TEXT',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'REAL'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_uap_id ON uap_reports(uap_id)")


# US Bigfoot Reports Table
bigfoot_df = pd.read_csv("../data/final/bigfoot_reports.csv")

bigfoot_df.to_sql(
    name='bigfoot_reports', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'bf_id': 'INTEGER',
        'full_date': 'TEXT', 
        'title': 'TEXT', 
        'state_code': 'TEXT', 
        'state': 'TEXT', 
        'latitude': 'REAL', 
        'longitude': 'REAL', 
        'geohash_5': 'TEXT', 
        'geohash_6': 'TEXT', 
        'geohash_7': 'TEXT', 
        'geohash': 'TEXT', 
        'date': 'TEXT', 
        'year': 'INTEGER', 
        'month': 'INTEGER', 
        'day': 'INTEGER', 
        'season': 'TEXT', 
        'temperature_mid': 'REAL', 
        'dew_point': 'REAL', 
        'cloud_cover': 'REAL', 
        'moon_phase': 'REAL', 
        'precip_type': 'TEXT', 
        'classification': 'TEXT', 
        'observed': 'TEXT',
        'solar_kp_index': 'REAL',
        'solar_ap_index': 'REAL'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_bf_id ON bigfoot_reports(bf_id)")



# States Table
states_df = pd.read_csv("../data/final/states.csv")

states_df.to_sql(
    name='states', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'state_code': 'TEXT',
        'state': 'TEXT', 
        '2010_population': 'INTEGER' 
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_state_code ON states(state_code)")


# Proximity Table
proximity_df = pd.read_csv("../data/final/proximity.csv")

proximity_df.to_sql(
    name='proximity', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'bf_id': 'INTEGER', 
        'uap_id': 'INTEGER', 
        'city': 'TEXT', 
        'state_code': 'TEXT', 
        'geohash_6': 'TEXT', 
        'geohash_7_bf': 'TEXT', 
        'geohash_7_uap': 'TEXT', 
        'latitude_bf': 'REAL', 
        'longitude_bf': 'REAL', 
        'latitude_uap': 'REAL', 
        'longitude_uap': 'REAL', 
        'full_date_bf': 'TEXT', 
        'full_date_uap': 'TEXT', 
        'date_diff_days': 'INTEGER',
        'year_bf': 'INTEGER', 
        'year_uap': 'INTEGER', 
        'distance_meters': 'REAL', 
        'proximity_score': 'REAL',
        'proximity_rank': 'INTEGER'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_proximity_pk ON proximity(bf_id, uap_id)")


# US UAP 2010-2014 Reports With Weather Enrichment
us_uap_2010_2014_weather_df = pd.read_csv("../data/final/us_uap_2010_2014_weather.csv")

us_uap_2010_2014_weather_df.to_sql(
    name='us_uap_2010_2014_weather', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'uap_id': 'INTEGER',
        'datetime': 'TEXT', 
        'city': 'TEXT', 
        'state_code': 'TEXT', 
        'state': 'TEXT', 
        'country': 'TEXT', 
        'shape': 'TEXT', 
        'duration (seconds)': 'INTEGER', 
        'duration (hours/min)': 'TEXT', 
        'comments': 'TEXT', 
        'date posted': 'TEXT', 
        'latitude': 'REAL', 
        'longitude': 'REAL', 
        'rounded_dt': 'TEXT', 
        'date_str': 'TEXT', 
        'hour_str': 'INTEGER', 
        'weather_cloud': 'REAL', 
        'weather_temp_f': 'REAL', 
        'weather_condition': 'TEXT', 
        'condition_groups': 'TEXT' 
    }
)
connection.execute("""
    CREATE INDEX IF NOT EXISTS idx_weather_uap_id 
    ON us_uap_2010_2014_weather(uap_id)
""")

# kp index table from processed/ folder
kp_index_df = pd.read_csv("../data/processed/kp_index.csv")

kp_index_df.to_sql(
    name='kp_index', 
    con=connection, 
    if_exists='replace', 
    index=False,
    dtype={
        'datetime': 'TEXT',
        'year': 'INTEGER', 
        'month': 'INTEGER', 
        'day': 'INTEGER', 
        'hour_start': 'REAL', 
        'hour_end': 'REAL', 
        'decimal_day_start': 'REAL',
        'decimal_day_end': 'REAL', 
        'kp': 'REAL', 'ap': 
        'INTEGER', 'flag': 'INTEGER'
    }
)
connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_datetime ON kp_index(datetime)")

print("\nAll tables created successfully!")



All tables created successfully!


In [21]:
query1 = """
SELECT name FROM sqlite_master 
WHERE type='table' 
AND name NOT LIKE 'sqlite_%';
"""
result1 = pd.read_sql(query1, connection)
result1

,name
0,uap_reports
1,bigfoot_reports
2,states
3,proximity
4,us_uap_2010_2014_weather
5,kp_index


In [22]:
# KP Index - Pecent of Reports Closest to KP Index vs KP Index Distribution 

kp_index_pct_query = """
WITH
    kp_stats AS (
        SELECT 
            ROUND(k.kp) AS rounded_kp,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS kp_percent
        FROM kp_index k
        GROUP BY ROUND(k.kp)
    ),
    bigfoot_kp AS (
        SELECT 
            ROUND(solar_kp_index) AS rounded_kp,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS bf_kp_percent
        FROM bigfoot_reports
        GROUP BY ROUND(solar_kp_index)
    ),
    uap_kp AS (
        SELECT 
            ROUND(solar_kp_index) AS rounded_kp,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS uap_kp_percent
        FROM uap_reports
        GROUP BY ROUND(solar_kp_index)
    )

SELECT
    k.rounded_kp AS kp_index,
    k.kp_percent AS kp_pct,
    b.bf_kp_percent AS bigfoot_kp_pct,
    u.uap_kp_percent AS uap_kp_pct,
    ROUND(b.bf_kp_percent - k.kp_percent, 2) AS bigfoot_kp_diff,
    ROUND(u.uap_kp_percent - k.kp_percent, 2) AS uap_kp_diff
FROM kp_stats k
LEFT JOIN bigfoot_kp b   ON k.rounded_kp = b.rounded_kp
LEFT JOIN uap_kp u       ON k.rounded_kp = u.rounded_kp
ORDER BY k.rounded_kp ASC;
"""


kp_index_pct_result = pd.read_sql(kp_index_pct_query, connection)
kp_index_pct_result


,kp_index,kp_pct,bigfoot_kp_pct,uap_kp_pct,bigfoot_kp_diff,uap_kp_diff
0,0.0,10.98,14.46,19.12,3.48,8.14
1,1.0,26.62,26.31,26.34,-0.31,-0.28
2,2.0,25.21,25.69,24.47,0.48,-0.74
3,3.0,19.52,18.55,16.93,-0.97,-2.59
4,4.0,10.74,9.05,8.08,-1.69,-2.66
5,5.0,4.54,4.17,3.30,-0.37,-1.24
6,6.0,1.56,1.41,1.20,-0.15,-0.36
7,7.0,0.56,0.20,0.35,-0.36,-0.21
8,8.0,0.22,0.10,0.14,-0.12,-0.08
9,9.0,0.05,0.07,0.05,0.02,0.00


In [23]:
states_query = """
WITH 
    states_pop AS (
        SELECT 
            state_code,
            "2010_population" as population
        FROM states
    ),
    bigfoot AS (
        SELECT
            state_code,
            COUNT(*) as bigfoot_total
        FROM bigfoot_reports
        GROUP BY state_code    
    ),
    uap AS (
        SELECT
            state_code,
            COUNT(*) as uap_total
        FROM uap_reports
        GROUP BY state_code    
    ),
    prox AS (
        SELECT
            state_code, 
            COUNT(*) as proximity_total
        FROM proximity
        GROUP BY state_code
    )
SELECT 
    s.state_code,
    s.population,
    COALESCE(b.bigfoot_total, 0) AS bigfoot_total,
    COALESCE(u.uap_total, 0) AS uap_total,
    COALESCE(p.proximity_total, 0) AS proximity_total,   
    ROUND(
        COALESCE(b.bigfoot_total, 0) * 1000000.0 / s.population, 
        2
    ) AS bigfoot_per_mil,
    ROUND(
        COALESCE(u.uap_total, 0) * 1000000.0 / s.population, 
        2
    ) AS uap_per_mil,
    ROUND(
        COALESCE(p.proximity_total, 0) * 1000000.0 / s.population, 
        2
    ) AS proximity_per_mil
FROM states_pop s
LEFT JOIN bigfoot b ON s.state_code = b.state_code
LEFT JOIN uap u ON s.state_code = u.state_code
LEFT JOIN prox p ON s.state_code = p.state_code
ORDER BY s.state_code; """

states_result = pd.read_sql(states_query, connection)
states_result


,state_code,population,bigfoot_total,uap_total,proximity_total,bigfoot_per_mil,uap_per_mil,proximity_per_mil
0,AK,710231,11,340,0,15.49,478.72,0.00
1,AL,4785514,86,705,7,17.97,147.32,1.46
2,AR,2921998,91,642,42,31.14,219.71,14.37
3,AZ,6407342,63,2617,1,9.83,408.44,0.16
4,CA,37319550,275,9574,598,7.37,256.54,16.02
5,CO,5047539,107,1520,6,21.20,301.14,1.19
6,CT,3579173,13,971,0,3.63,271.29,0.00
7,DC,605282,0,7,0,0.00,11.56,0.00
8,DE,899647,3,180,0,3.33,200.08,0.00
9,FL,18846143,280,4155,87,14.86,220.47,4.62


In [ ]:
# high strangeness factor